In [1]:
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../hayoung/finetune_v1/code")
from query_parser import parse_query

sys.path.insert(0, "../../backend")
from pickcu.filter_engine import recommend


def recommend_from_text(text, categories):
    """문장을 해석해 필터 엔진의 Top 5 추천을 돌려준다. 해석에 실패하면 None."""
    query = parse_query(text)
    if query is None:
        return None
    return {"query": query, "results": recommend(categories, query["nutrients"], budget=query["budget"])}


def to_dataframe(results):
    """추천 결과를 상품 한 개가 한 행인 DataFrame으로 바꾼다."""
    rows = []
    for result in results:
        for item in result["items"]:
            rows.append({
                "순위": result["rank"],
                "총점": result["total_score"],
                "영양점수": result["nutrition_score"],
                "가격점수": result["price_score"],
                "합계": result["total_price"],
                "카테고리": item["category"],
                "상품명": item["product_name"],
                "가격": item["price"],
                "행사": "" if item["event_type"] == "NONE" else item["event_type"],
                "상품영양점수": item["nutrition_score"],
                "열량": item["calories"],
                "단백질": item["protein"],
                "나트륨": item["sodium"],
                "탄수화물": item["carbs"],
                "당류": item["sugar"],
                "지방": item["fat"],
            })
    return pd.DataFrame(rows)


def show_recommend(text, categories):
    """문장 하나의 추천 결과를 표로 보여 주고 DataFrame을 돌려준다. 결과가 없으면 None."""
    print(f"입력: {text} / 카테고리: {categories}")
    output = recommend_from_text(text, categories)
    if output is None:
        print("문장 해석 실패\n")
        return None
    print(f"추출: {output['query']}")
    if not output["results"]:
        print("조건에 맞는 조합 없음\n")
        return None
    df = to_dataframe(output["results"])
    display(df)
    return df

In [2]:
# 시험할 문장과 카테고리(화면에서 고르는 값, 1~3개)를 여기에 추가한다.
tests = [
    ("5천원 이하로 운동 후에 먹을 거", ["간편식사", "음료"]),
    ("야식으로 먹을 건데 2만원대로 짜지 않은 거", ["즉석조리", "음료"]),
    ("요즘 혈당이 걱정돼요", ["과자류"]),
]

frames = [show_recommend(text, categories) for text, categories in tests]

입력: 5천원 이하로 운동 후에 먹을 거 / 카테고리: ['간편식사', '음료']


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


추출: {'budget': 5000, 'nutrients': {'단백질': 'high', '지방': 'low'}}


,순위,총점,영양점수,가격점수,합계,카테고리,상품명,가격,행사,상품영양점수,열량,단백질,나트륨,탄수화물,당류,지방
0,1,82.3,75.5,98.0,4900,간편식사,샌)급식대가저당치킨샌드,3600,,90.2,338.0,26.0,810.0,45.0,4.0,6.0
1,1,82.3,75.5,98.0,4900,음료,레쓰비)캔200ml,1300,,60.9,60.0,1.0,95.0,14.0,14.0,0.0
2,2,79.4,70.6,100.0,5000,간편식사,샌)호밀햄치즈샌드,3300,,80.2,318.0,16.0,1200.0,50.0,10.0,6.0
3,2,79.4,70.6,100.0,5000,음료,매일)엔요280ml,1700,2+1,60.9,40.0,1.0,20.0,9.0,6.0,0.0
4,3,78.2,68.8,100.0,5000,간편식사,빅삼)정통돈까스삼각,1700,,54.0,289.0,7.0,630.0,54.0,2.0,5.0
5,3,78.2,68.8,100.0,5000,음료,매일)셀렉스프로핏코코넛,3300,,83.7,90.0,20.0,290.0,4.5,0.0,0.0
6,4,77.9,68.5,100.0,5000,간편식사,삼)떡갈비전주비빔삼각,1500,,55.9,166.0,5.0,490.0,32.0,4.0,2.0
7,4,77.9,68.5,100.0,5000,음료,유한)비컴플런부스터110ml,3500,1+1,81.1,92.0,9.0,10.0,14.0,0.0,0.0
8,5,76.5,66.5,100.0,5000,간편식사,주)매콤양념족발밥바,2600,,56.2,299.0,10.0,820.0,49.0,4.0,7.0
9,5,76.5,66.5,100.0,5000,음료,풀무원)액티비아업딸기,2400,1+1,76.8,105.0,6.0,70.0,20.0,15.0,0.0


입력: 야식으로 먹을 건데 2만원대로 짜지 않은 거 / 카테고리: ['즉석조리', '음료']


추출: {'budget': 20000, 'nutrients': {'나트륨': 'low', '지방': 'low'}}


,순위,총점,영양점수,가격점수,합계,카테고리,상품명,가격,행사,상품영양점수,열량,단백질,나트륨,탄수화물,당류,지방
0,1,79.3,85.0,66.0,13200,즉석조리,프라페)밀크믹스프레소,3200,,96.1,200.0,4.0,130.0,35.0,34.0,4.9
1,1,79.3,85.0,66.0,13200,음료,HK)컨디션CEO병,10000,2+1,73.9,80.0,2.0,20.0,18.0,17.0,0.0
2,2,72.4,90.3,30.5,6100,즉석조리,오뎅)물떡꼬치,1100,,96.1,195.0,4.1,136.3,44.2,0.0,0.2
3,2,72.4,90.3,30.5,6100,음료,삼양사)상쾌환부스터제로,5000,1+1,84.6,0.0,0.0,0.0,0.0,0.0,0.0
4,3,68.8,82.7,36.5,7300,즉석조리,튀김)애플파이,2000,2+1,83.6,196.0,2.0,160.0,29.0,14.0,8.0
5,3,68.8,82.7,36.5,7300,음료,코카)미닛오렌지P1.5L,5300,,81.8,30.0,0.4,5.0,7.0,6.0,0.0
6,4,65.4,79.7,32.0,6400,즉석조리,도너츠)스위트츄잉도넛N,1200,,80.3,220.0,1.0,125.0,25.0,7.0,12.0
7,4,65.4,79.7,32.0,6400,음료,웅진)자연은망고1.5L,5200,,79.1,50.0,0.0,10.0,12.0,12.0,0.0
8,5,64.6,80.9,26.5,5300,즉석조리,도너츠)초코후로스트N,1100,,78.9,200.0,4.0,145.0,23.0,4.0,10.0
9,5,64.6,80.9,26.5,5300,음료,롯데)칠성사이다제로P1.5,4200,1+1,82.8,0.0,0.0,3.0,1.0,1.0,0.0


입력: 요즘 혈당이 걱정돼요 / 카테고리: ['과자류']


추출: {'budget': None, 'nutrients': {'당류': 'low', '열량': 'low'}}


,순위,총점,영양점수,가격점수,합계,카테고리,상품명,가격,행사,상품영양점수,열량,단백질,나트륨,탄수화물,당류,지방
0,1,99.0,99.0,None,900,과자류,MDS)젤리블리애플샤인,900,,99.0,0.0,0.0,40.0,9.0,0.0,0.0
1,2,98.6,98.6,None,3500,과자류,보람)쥬쥬마이크노래캔디,3500,,98.6,24.0,0.0,2.0,6.0,0.0,0.0
2,3,98.5,98.5,None,1200,과자류,위스트)마라곤약향라맛,1200,1+1,98.5,25.0,0.0,235.0,3.0,0.0,1.4
3,4,98.1,98.1,None,3500,과자류,스위트)토이스토리히퍼,3500,2+1,98.1,40.0,0.0,4.0,10.0,0.0,0.0
4,5,98.1,98.1,None,3500,과자류,스위트)미니언즈랜덤캔디,3500,,98.1,40.0,0.0,4.0,10.0,0.0,0.0


In [3]:
# 시험할 문장과 카테고리(화면에서 고르는 값, 1~3개)를 여기에 추가한다.
tests = [
    # ("더운데 천원짜리 두 장으로 살 수 있는 아이스크림 뭐야?", ["간편식사", "음료"]),
    # ("눈 뜨자마자 천원짜리 다섯 장 들고 나왔어. 아침으로 뭘 사면 든든할까?", ["즉석조리", "음료"])
    ("안녕하세요!", ["즉석조리", "음료"])
]

frames = [show_recommend(text, categories) for text, categories in tests]

영양 조건이 없는 입력(인삿말·잡담)이라 모델을 부르지 않음: 안녕하세요!


입력: 안녕하세요! / 카테고리: ['즉석조리', '음료']
문장 해석 실패

